In [1]:
# Source - https://stackoverflow.com/a/69297835
# Posted by jmPicaza
# Retrieved 2026-01-17, License - CC BY-SA 4.0

from IPython.core.display import HTML
display(HTML("<style>pre { white-space: pre !important; }</style>"))


In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.appName('iceberg-incremenatal')
    
    # Iceberg package (same as --packages)
    .config(
        "spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.9.2"
    )
    
    # Catalog configuration
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.local.type", "hadoop")
    .config("spark.sql.catalog.local.warehouse", "/Users/codebase/Documents/codebase/Courses/LearnApacheIceberg/warehouse")
        
    .getOrCreate()
    )

spark

26/01/18 12:41:52 WARN Utils: Your hostname, codebase.local resolves to a loopback address: 127.0.0.1; using 192.168.1.7 instead (on interface en0)
26/01/18 12:41:52 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /Users/codebase/.ivy2/cache
The jars for the packages stored in: /Users/codebase/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-59e0de9c-f07f-4b8d-b7af-50581cbfc61b;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.9.2 in central
:: resolution report :: resolve 39ms :: artifacts dl 1ms
	:: modules in use:
	org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.9.2 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	-

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/01/18 12:42:11 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [26]:
from datetime import datetime as dt

data_to_write = []
data = (2, "Charlie", 28, dt.now())
for i in range(1000_000):
    data_to_write.append(data)

In [27]:
df = spark.createDataFrame(data_to_write, ["id", "name", "age", "ts"])

df.show(5)

+---+-------+---+--------------------+
| id|   name|age|                  ts|
+---+-------+---+--------------------+
|  2|Charlie| 28|2026-01-18 12:52:...|
|  2|Charlie| 28|2026-01-18 12:52:...|
|  2|Charlie| 28|2026-01-18 12:52:...|
|  2|Charlie| 28|2026-01-18 12:52:...|
|  2|Charlie| 28|2026-01-18 12:52:...|
+---+-------+---+--------------------+
only showing top 5 rows



26/01/18 12:52:39 WARN TaskSetManager: Stage 26 contains a task of very large size (1758 KiB). The maximum recommended task size is 1000 KiB.


In [28]:
df.count()

26/01/18 12:52:39 WARN TaskSetManager: Stage 27 contains a task of very large size (1758 KiB). The maximum recommended task size is 1000 KiB.


1000000

In [29]:
df.writeTo("local.db.users").append()

26/01/18 12:52:39 WARN TaskSetManager: Stage 30 contains a task of very large size (1758 KiB). The maximum recommended task size is 1000 KiB.


In [30]:
from pyspark.sql.functions import col, max as spark_max

def iceberg_max_from_stats(table, ts_cols):
    """
    table   : 'local.db.events'
    ts_cols : ['ts', 'ingest_ts']
    """

    df = spark.table(f"{table}.files")

    return df.select([
        spark_max(
            col("readable_metrics")
            .getItem(c)
            .getField("upper_bound")
        ).alias(f"max_{c}")
        for c in ts_cols
    ])

In [31]:
iceberg_max_from_stats(
    table='local.db.users',
    ts_cols=['ts', 'id']
).show(truncate=False)

+-------------------------+------+
|max_ts                   |max_id|
+-------------------------+------+
|2026-01-18 12:52:31.66519|2     |
+-------------------------+------+



In [32]:
spark.table("local.db.events.files").show(200, truncate=False)

+-------+-----------------------------------------------------------------------------------------------------------------------------------------------------+-----------+-------+------------+------------------+----------------------------------------+----------------------------------------------------+--------------------------------+----------------+-----------------------------------------------------------------------------------------------------------------------------+-----------------------------------------------------------------------------------------------------------------------------+------------+-------------+------------+-------------+------------+--------------------+--------------+---------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|content|file_path                                                                

In [36]:
spark.sql(
    """
    UPDATE local.db.users
    SET name = 'Rohit'
    WHERE id = 1;
    """
)

DataFrame[]

In [48]:
spark.table(
    "local.db.users.files"
).selectExpr("SUM(record_count) AS row_count").collect()[0]['row_count']

3000000

In [39]:
from pyspark.sql.functions import sum as spark_sum

spark.sql("""
SELECT SUM(record_count) AS total_rows
FROM local.db.users.files
""").show()

+----------+
|total_rows|
+----------+
|   3000000|
+----------+

